# Nova App Analytics

## Market Strategic Archetypes: Clustering

This notebook segments Nova markets into strategic archetypes for the geography dashboard.

Unlike the regression notebook, clustering has **no target variable**. The goal is to group markets that look similar across scale, growth, supply, reliability, macro context, category mix, and weather sensitivity.

The output should complement the existing dashboard story:

- Opportunity score ranks markets.
- Regression explains demand predictors.
- Rain lift explains category weather sensitivity.
- Clustering groups markets into similar strategic profiles.

## 1. Business Problem and Clustering Framing

The business question is:

> Which Nova markets behave similarly, and how can we describe those market groups for strategy and dashboard storytelling?

This is an **unsupervised learning** problem.

- There is no `y` target.
- The model groups markets based on feature similarity.
- The cluster labels are analyst interpretation, not model truth.
- Because there are only 16 markets, the result should be treated as a simple segmentation aid, not a definitive statistical classification.

## 2. Setup

In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd
import pandas_gbq
import plotly.express as px
import plotly.io as pio

from google.cloud import bigquery
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", "{:,.4f}".format)

px.defaults.template = "plotly_white"
px.defaults.width = 1_050
px.defaults.height = 560

# If charts do not render in your environment, try:
# pio.renderers.default = "browser"
pio.renderers.default = "notebook_connected"

## 3. Load the Market Cluster Feature Mart

The source table is the dbt mart built specifically for clustering:

`mart_geo_market_cluster_features`

Expected grain:

```text
one row = one market
```

In [2]:
PROJECT_ID = "nova-project-498911"
LOCATION = "EU"
DATASET_ID = "dbt_doruk"
TABLE_NAME = "mart_geo_market_cluster_features"

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)


def table_ref(table_name: str) -> str:
    return f"`{PROJECT_ID}.{DATASET_ID}.{table_name}`"


def read_bq(sql: str) -> pd.DataFrame:
    return client.query(sql).to_dataframe()


sql = f"""
select *
from {table_ref(TABLE_NAME)}
order by market_id
"""

markets = read_bq(sql)

print(f"Rows: {markets.shape[0]:,}")
print(f"Columns: {markets.shape[1]:,}")

markets.head()

/Users/doruk/dev/nova-analytics/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Rows: 16
Columns: 83


,market_id,market_name,nova_region,country_name,country_iso3,country_region,country_subregion,latitude,longitude,total_transactions,total_gmv_usd,avg_daily_transactions,avg_daily_gmv_usd,avg_transaction_amount_usd,annual_active_users,market_active_user_penetration,market_urban_active_user_penetration,country_active_user_penetration,country_urban_active_user_penetration,transaction_growth_rate,gmv_growth_rate,completion_rate,failed_rate,refunded_rate,promo_share,ios_transaction_share,android_transaction_share,lite_transaction_share,web_transaction_share,active_services,avg_service_rating,avg_popularity_score,head_services,mid_services,long_tail_services,head_service_share,mid_service_share,long_tail_service_share,population_total,gdp_per_capita_current_usd,urban_population_pct,internet_users_pct,mobile_subscriptions_per_100_people,market_weight,growth_multiplier,configured_ios_share,current_performance_score,growth_score,macro_potential_score,reliability_score,supply_score,adoption_score,opportunity_score,opportunity_rank,opportunity_segment,food_delivery_transaction_share,ride_hailing_transaction_share,e_commerce_transaction_share,grocery_transaction_share,digital_wallet_transaction_share,food_delivery_gmv_share,ride_hailing_gmv_share,e_commerce_gmv_share,grocery_gmv_share,digital_wallet_gmv_share,avg_temperature_2m_mean_c,avg_precipitation_sum_mm,avg_rain_sum_mm,avg_precipitation_hours,avg_wind_speed_10m_max_kmh,rain_day_share,avg_rain_transaction_lift_pct,avg_rain_gmv_lift_pct,max_abs_rain_transaction_lift_pct,max_abs_rain_gmv_lift_pct,avg_precipitation_transaction_corr,avg_temperature_transaction_corr,rain_positive_categories,rain_negative_categories,low_rain_sensitivity_categories,limited_comparison_categories,rain_positive_category_share,rain_negative_category_share
0,1,Singapore,Southeast Asia,Singapore,SGP,Asia,South-Eastern Asia,1.3521,103.8198,3338833,"217,194,167.9400","9,122.4945","593,426.6884",65.0509,119016,0.0197,0.0197,0.0197,0.0197,0.7005,0.8830,0.9164,0.0452,0.0383,0.1909,0.4294,0.2723,0.1143,0.1840,3537,4.3842,2.9074,78,612,2847,0.0221,0.1730,0.8049,"6,036,860.0000","90,674.0666",100.0000,94.3776,170.7826,0.0700,1.0800,0.5500,68.2491,70.0493,71.5565,91.6439,48.0147,100.0000,71.3800,1,Growth candidate,0.2946,0.1856,0.2441,0.1590,0.1168,0.1560,0.0924,0.4352,0.2159,0.1005,26.7866,10.5298,10.5298,8.5027,13.5317,0.9754,0.0469,0.0560,0.0917,0.1120,0.0074,-0.1152,0,0,0,5,0.0000,0.0000
1,2,Jakarta,Southeast Asia,Indonesia,IDN,Asia,South-Eastern Asia,-6.2088,106.8456,4732472,"202,985,200.9800","12,930.2514","554,604.3743",42.8920,153953,0.0005,0.0009,0.0005,0.0009,0.6925,0.8524,0.9064,0.0552,0.0385,0.1961,0.2228,0.4223,0.1812,0.1737,4550,4.3916,2.7337,79,834,3637,0.0174,0.1833,0.7993,"283,487,931.0000","4,925.4305",58.7510,72.7808,122.5149,0.0900,1.1800,0.2800,90.8554,69.2541,23.0222,90.6369,77.7036,2.2819,53.5500,2,Maintain,0.3224,0.2380,0.1795,0.1240,0.1361,0.1957,0.1386,0.3467,0.1834,0.1356,27.3967,6.8311,6.8311,7.4590,12.8057,0.8579,-0.0169,-0.0093,0.0602,0.0878,-0.0623,0.1847,0,0,5,0,0.0000,0.0000
2,3,Manila,Southeast Asia,Philippines,PHL,Asia,South-Eastern Asia,14.5995,120.9842,3873921,"177,769,906.4700","10,584.4836","485,710.1270",45.8889,128517,0.0011,0.0020,0.0011,0.0020,0.6977,0.8564,0.9075,0.0532,0.0393,0.1959,0.2572,0.3974,0.1703,0.1751,3808,4.3988,2.9740,92,656,3060,0.0242,0.1723,0.8036,"115,843,670.0000","3,984.8315",55.4521,67.2630,115.2987,0.0750,1.1600,0.3200,69.7141,69.7686,15.1537,90.7508,80.6785,6.5181,45.6300,5,Maintain,0.3371,0.2144,0.1940,0.1256,0.1289,0.2019,0.1203,0.3704,0.1826,0.1248,28.0230,6.3082,6.3082,7.2732,16.6511,0.7896,0.0962,0.0964,0.1220,0.1251,0.1386,-0.1280,2,0,3,0,0.4000,0.0000
3,4,Bangkok,Southeast Asia,Thailand,THA,Asia,South-Eastern Asia,13.7563,100.5018,3267079,"152,273,022.5000","8,926.4454","416,046.5096",46.6083,110905,0.0015,0.0025,0.0015,0.0025,0.6905,0.8558,0.9101,0.0518,0.0381,0.1987,0.2882,0.3750,0.1602,0.1765,3247,4.3910,2.8002,64,601,2582,0.0197,0.1851,0.7952,"71,

## 4. Basic Checks

Before clustering, confirm that the mart has one row per market and the expected fields are present.

In [3]:
required_columns = ["market_id", "market_name", "nova_region", "total_transactions", "active_services", "opportunity_score"]
missing_required = [col for col in required_columns if col not in markets.columns]

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

duplicate_markets = int(markets.duplicated(["market_id"]).sum())
print(f"Duplicate market rows: {duplicate_markets:,}")
if duplicate_markets > 0:
    raise ValueError("The clustering mart must have one row per market.")

overview = pd.DataFrame(
    {
        "metric": ["markets", "regions", "countries"],
        "value": [
            markets["market_id"].nunique(),
            markets["nova_region"].nunique(),
            markets["country_iso3"].nunique(),
        ],
    }
)

display(overview)

missing_summary = (
    markets.isna()
    .sum()
    .to_frame("missing_values")
    .query("missing_values > 0")
    .sort_values("missing_values", ascending=False)
)

missing_summary

Duplicate market rows: 0


,metric,value
0,markets,16
1,regions,8
2,countries,15


,missing_values


## 5. Select Clustering Features

For clustering, we intentionally include current performance outcomes such as transactions and GMV. This is different from supervised prediction: the goal here is descriptive segmentation, not forecasting.

Feature groups:

- Demand scale and growth
- Reliability and platform mix
- Service supply
- Macro and market context
- Opportunity scores
- Category mix
- Weather sensitivity

In [4]:
id_cols = [
    "market_id",
    "market_name",
    "nova_region",
    "country_name",
    "country_iso3",
    "opportunity_segment",
]

clustering_features = [
    "total_transactions",
    "total_gmv_usd",
    "avg_daily_transactions",
    "avg_daily_gmv_usd",
    "avg_transaction_amount_usd",
    "annual_active_users",
    "market_active_user_penetration",
    "transaction_growth_rate",
    "gmv_growth_rate",
    "completion_rate",
    "failed_rate",
    "refunded_rate",
    "promo_share",
    "ios_transaction_share",
    "android_transaction_share",
    "lite_transaction_share",
    "web_transaction_share",
    "active_services",
    "avg_service_rating",
    "avg_popularity_score",
    "head_service_share",
    "mid_service_share",
    "long_tail_service_share",
    "population_total",
    "gdp_per_capita_current_usd",
    "urban_population_pct",
    "internet_users_pct",
    "mobile_subscriptions_per_100_people",
    "market_weight",
    "growth_multiplier",
    "current_performance_score",
    "growth_score",
    "macro_potential_score",
    "reliability_score",
    "supply_score",
    "adoption_score",
    "opportunity_score",
    "food_delivery_transaction_share",
    "ride_hailing_transaction_share",
    "e_commerce_transaction_share",
    "grocery_transaction_share",
    "digital_wallet_transaction_share",
    "avg_temperature_2m_mean_c",
    "avg_precipitation_sum_mm",
    "rain_day_share",
    "avg_rain_transaction_lift_pct",
    "max_abs_rain_transaction_lift_pct",
    "rain_positive_category_share",
    "rain_negative_category_share",
]

available_features = [col for col in clustering_features if col in markets.columns]
missing_features = [col for col in clustering_features if col not in markets.columns]

print(f"Available clustering features: {len(available_features)}")
print(f"Missing requested features: {missing_features}")

feature_input = markets[id_cols + available_features].copy()
feature_input.head()

Available clustering features: 49
Missing requested features: []


,market_id,market_name,nova_region,country_name,country_iso3,opportunity_segment,total_transactions,total_gmv_usd,avg_daily_transactions,avg_daily_gmv_usd,avg_transaction_amount_usd,annual_active_users,market_active_user_penetration,transaction_growth_rate,gmv_growth_rate,completion_rate,failed_rate,refunded_rate,promo_share,ios_transaction_share,android_transaction_share,lite_transaction_share,web_transaction_share,active_services,avg_service_rating,avg_popularity_score,head_service_share,mid_service_share,long_tail_service_share,population_total,gdp_per_capita_current_usd,urban_population_pct,internet_users_pct,mobile_subscriptions_per_100_people,market_weight,growth_multiplier,current_performance_score,growth_score,macro_potential_score,reliability_score,supply_score,adoption_score,opportunity_score,food_delivery_transaction_share,ride_hailing_transaction_share,e_commerce_transaction_share,grocery_transaction_share,digital_wallet_transaction_share,avg_temperature_2m_mean_c,avg_precipitation_sum_mm,rain_day_share,avg_rain_transaction_lift_pct,max_abs_rain_transaction_lift_pct,rain_positive_category_share,rain_negative_category_share
0,1,Singapore,Southeast Asia,Singapore,SGP,Growth candidate,3338833,"217,194,167.9400","9,122.4945","593,426.6884",65.0509,119016,0.0197,0.7005,0.8830,0.9164,0.0452,0.0383,0.1909,0.4294,0.2723,0.1143,0.1840,3537,4.3842,2.9074,0.0221,0.1730,0.8049,"6,036,860.0000","90,674.0666",100.0000,94.3776,170.7826,0.0700,1.0800,68.2491,70.0493,71.5565,91.6439,48.0147,100.0000,71.3800,0.2946,0.1856,0.2441,0.1590,0.1168,26.7866,10.5298,0.9754,0.0469,0.0917,0.0000,0.0000
1,2,Jakarta,Southeast Asia,Indonesia,IDN,Maintain,4732472,"202,985,200.9800","12,930.2514","554,604.3743",42.8920,153953,0.0005,0.6925,0.8524,0.9064,0.0552,0.0385,0.1961,0.2228,0.4223,0.1812,0.1737,4550,4.3916,2.7337,0.0174,0.1833,0.7993,"283,487,931.0000","4,925.4305",58.7510,72.7808,122.5149,0.0900,1.1800,90.8554,69.2541,23.0222,90.6369,77.7036,2.2819,53.5500,0.3224,0.2380,0.1795,0.1240,0.1361,27.3967,6.8311,0.8579,-0.0169,0.0602,0.0000,0.0000
2,3,Manila,Southeast Asia,Philippines,PHL,Maintain,3873921,"177,769,906.4700","10,584.4836","485,710.1270",45.8889,128517,0.0011,0.6977,0.8564,0.9075,0.0532,0.0393,0.1959,0.2572,0.3974,0.1703,0.1751,3808,4.3988,2.9740,0.0242,0.1723,0.8036,"115,843,670.0000","3,984.8315",55.4521,67.2630,115.2987,0.0750,1.1600,69.7141,69.7686,15.1537,90.7508,80.6785,6.5181,45.6300,0.3371,0.2144,0.1940,0.1256,0.1289,28.0230,6.3082,0.7896,0.0962,0.1220,0.4000,0.0000
3,4,Bangkok,Southeast Asia,Thailand,THA,Monitor,3267079,"152,273,022.5000","8,926.4454","416,046.5096",46.6083,110905,0.0015,0.6905,0.8558,0.9101,0.0518,0.0381,0.1987,0.2882,0.3750,0.1602,0.1765,3247,4.3910,2.8002,0.0197,0.1851,0.7952,"71,668,011.0000","7,346.6202",61.8689,90.8672,160.6393,0.0650,1.1200,53.1907,69.0500,38.0804,91.0070,57.0388,8.9354,41.3300,0.3429,0.2207,0.1901,0.1309,0.1154,28.7918,4.4727,0.6858,-0.0072,0.0833,0.0000,0.0000
4,5,Ho Chi Minh City,Southeast Asia,Vietnam,VNM,Monitor,3090678,"124,719,369.5600","8,444.4754","340,763.3048",40.3534,102651,0.0010,0.6980,0.8756,0.9124,0.0513,0.0363,0.1988,0.2388,0.4111,0.1760,0.1741,3000,4.3789,2.6937,0.0173,0.1857,0.7970,"100,987,686.0000","4,717.2903",38.4901,84.1500,127.6065,0.0600,1.1700,44.0717,69.8009,21.4865,91.2371,29.5711,7.9381,28.3000,0.3488,0.2256,0.1752,0.1210,0.1295,28.1260,5.8052,0.7158,0.1224,0.1567,0.8000,0.0000


## 6. Scale Features

KMeans uses distance. Without scaling, large-unit fields like population or GMV would dominate the clustering.

We impute missing numeric values with the median and standardize all selected features.

In [5]:
feature_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

X = feature_pipeline.fit_transform(feature_input[available_features])

scaled_features = pd.DataFrame(X, columns=available_features)
scaled_features.head()

,total_transactions,total_gmv_usd,avg_daily_transactions,avg_daily_gmv_usd,avg_transaction_amount_usd,annual_active_users,market_active_user_penetration,transaction_growth_rate,gmv_growth_rate,completion_rate,failed_rate,refunded_rate,promo_share,ios_transaction_share,android_transaction_share,lite_transaction_share,web_transaction_share,active_services,avg_service_rating,avg_popularity_score,head_service_share,mid_service_share,long_tail_service_share,population_total,gdp_per_capita_current_usd,urban_population_pct,internet_users_pct,mobile_subscriptions_per_100_people,market_weight,growth_multiplier,current_performance_score,growth_score,macro_potential_score,reliability_score,supply_score,adoption_score,opportunity_score,food_delivery_transaction_share,ride_hailing_transaction_share,e_commerce_transaction_share,grocery_transaction_share,digital_wallet_transaction_share,avg_temperature_2m_mean_c,avg_precipitation_sum_mm,rain_day_share,avg_rain_transaction_lift_pct,max_abs_rain_transaction_lift_pct,rain_positive_category_share,rain_negative_category_share
0,0.2685,1.1111,0.2685,1.1111,0.5285,0.5154,3.6008,-0.4172,-0.3030,1.3569,-1.2829,-0.9110,-0.8135,0.7688,-0.7679,-0.7705,0.7676,0.5739,-0.9572,0.7728,0.8501,-0.8530,0.6090,-0.6125,2.0844,1.3329,0.7716,1.2105,0.5345,-0.7652,0.7512,-0.4172,1.7338,1.3569,-0.4239,3.5602,2.6176,0.0334,-1.9498,0.5497,1.1244,-0.0416,0.7351,2.4979,1.7709,0.5869,0.0513,-0.5680,0.0000
1,2.0182,0.8030,2.0182,0.8030,-0.7361,1.9717,-0.4241,-0.6393,-0.8689,-0.5535,0.9305,-0.8217,0.5750,-1.1916,1.1900,1.1835,-1.1180,1.9848,0.0732,-0.6971,-0.9022,0.4903,-0.1912,0.0029,-0.8544,-0.6472,-1.0802,-0.2014,1.9599,1.1184,1.8953,-0.6393,-1.0331,-0.5535,1.3058,-0.4355,0.9946,0.9154,0.7890,-1.3024,-1.0955,1.2224,0.8387,1.0386,1.2645,-0.5197,-0.8532,-0.5680,0.0000
2,0.9403,0.2562,0.9403,0.2562,-0.5651,0.9115,-0.3052,-0.4956,-0.7960,-0.3375,0.4975,-0.2961,0.5212,-0.8649,0.8645,0.8659,-0.8648,0.9513,1.0767,1.3368,1.6373,-0.9523,0.4156,-0.3689,-0.8866,-0.8056,-1.5533,-0.4124,0.8909,0.7417,0.8253,-0.4956,-1.4816,-0.3375,1.4791,-0.2623,0.2737,1.3808,-0.4448,-0.8856,-0.9910,0.7492,0.9450,0.8323,0.9701,1.4434,0.9219,1.0844,0.0000
3,0.1784,-0.2967,0.1784,-0.2967,-0.5240,0.1774,-0.2132,-0.6962,-0.8064,0.1486,0.1813,-1.0427,1.2672,-0.5706,0.5726,0.5709,-0.6010,0.1699,-0.0059,-0.1339,-0.0250,0.7254,-0.7858,-0.4669,-0.7714,-0.4976,0.4706,0.9138,0.1782,-0.0118,-0.0110,-0.6962,-0.1746,0.1486,0.1018,-0.1635,-0.1177,1.5647,-0.1142,-0.9985,-0.6540,-0.1323,1.0755,0.1081,0.5226,-0.3511,-0.1907,-0.5680,0.0000
4,-0.0431,-0.8942,-0.0431,-0.8942,-0.8810,-0.1667,-0.3247,-0.4866,-0.4407,0.5852,0.0796,-2.2471,1.2990,-1.0398,1.0437,1.0317,-1.0422,-0.1741,-1.6959,-1.0350,-0.9131,0.8003,-0.5270,-0.4019,-0.8615,-1.6198,-0.1053,-0.0524,-0.1782,0.9301,-0.4725,-0.4866,-1.1206,0.5852,-1.4985,-0.2043,-1.3038,1.7518,0.1397,-1.4247,-1.2869,0.7873,0.9625,0.6338,0.6521,1.8969,1.9180,2.7369,0.0000


## 7. Evaluate Candidate Cluster Counts

We compare `k=3`, `k=4`, and `k=5`.

- **Inertia** decreases as clusters increase, so it is mainly useful for spotting diminishing returns.
- **Silhouette score** measures how separated clusters are. Higher is better.

With only 16 markets, we default to `k=4` unless the diagnostics are clearly poor.

In [6]:
candidate_k = [3, 4, 5]
k_results = []

for k in candidate_k:
    model = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = model.fit_predict(X)
    k_results.append(
        {
            "k": k,
            "inertia": model.inertia_,
            "silhouette_score": silhouette_score(X, labels),
        }
    )

k_results_df = pd.DataFrame(k_results)
display(k_results_df)

fig = px.line(
    k_results_df,
    x="k",
    y="silhouette_score",
    markers=True,
    title="Silhouette Score by Cluster Count",
    labels={"k": "Number of clusters", "silhouette_score": "Silhouette score"},
)
fig.show()

fig = px.line(
    k_results_df,
    x="k",
    y="inertia",
    markers=True,
    title="KMeans Inertia by Cluster Count",
    labels={"k": "Number of clusters", "inertia": "Inertia"},
)
fig.show()

,k,inertia,silhouette_score
0,3,414.2101,0.2244
1,4,361.3748,0.1789
2,5,273.6794,0.1982


## 8. Fit Final Clustering Model

We use `k=4` for the first dashboard version. This gives enough variation for strategy without creating tiny one-market segments by default.

In [7]:
FINAL_K = 4

kmeans = KMeans(n_clusters=FINAL_K, random_state=42, n_init="auto")
cluster_ids = kmeans.fit_predict(X)

clustered_markets = markets.copy()
clustered_markets["cluster_id"] = cluster_ids

clustered_markets[["market_id", "market_name", "nova_region", "opportunity_segment", "cluster_id"]].sort_values("cluster_id")

,market_id,market_name,nova_region,opportunity_segment,cluster_id
3,4,Bangkok,Southeast Asia,Monitor,0
4,5,Ho Chi Minh City,Southeast Asia,Monitor,0
5,6,Istanbul,EMEA,Monitor,0
6,7,Dubai,EMEA,Maintain,0
7,8,Riyadh,EMEA,Monitor,0
10,11,Mexico City,Latin America,Monitor,0
11,12,Sao Paulo,Latin America,Monitor,0
15,16,Sydney,Oceania,Monitor,0
1,2,Jakarta,Southeast Asia,Maintain,1
2,3,Manila,Southeast Asia,Maintain,1


## 9. Profile Clusters

Cluster IDs are arbitrary numbers. We inspect each cluster and then assign analyst-friendly labels.

In [8]:
cluster_profile = (
    clustered_markets.groupby("cluster_id", as_index=False)
    .agg(
        market_count=("market_id", "nunique"),
        total_transactions=("total_transactions", "sum"),
        total_gmv_usd=("total_gmv_usd", "sum"),
        avg_daily_transactions=("avg_daily_transactions", "mean"),
        transaction_growth_rate=("transaction_growth_rate", "mean"),
        completion_rate=("completion_rate", "mean"),
        active_services=("active_services", "mean"),
        supply_score=("supply_score", "mean"),
        macro_potential_score=("macro_potential_score", "mean"),
        opportunity_score=("opportunity_score", "mean"),
        rain_day_share=("rain_day_share", "mean"),
        avg_rain_transaction_lift_pct=("avg_rain_transaction_lift_pct", "mean"),
    )
    .sort_values("avg_daily_transactions", ascending=False)
)

display(cluster_profile)

cluster_members = (
    clustered_markets.groupby("cluster_id")
    .agg(markets=("market_name", lambda values: ", ".join(sorted(values))))
    .reset_index()
)

cluster_profile_with_members = cluster_profile.merge(cluster_members, on="cluster_id", how="left")
cluster_profile_with_members

,cluster_id,market_count,total_transactions,total_gmv_usd,avg_daily_transactions,transaction_growth_rate,completion_rate,active_services,supply_score,macro_potential_score,opportunity_score,rain_day_share,avg_rain_transaction_lift_pct
1,1,4,16675254,"660,253,581.3500","11,390.2008",0.6958,0.9040,"3,943.2500",74.5172,19.5440,45.5975,0.6653,0.0534
2,2,1,3338833,"217,194,167.9400","9,122.4945",0.7005,0.9164,"3,537.0000",48.0147,71.5565,71.3800,0.9754,0.0469
3,3,3,8606357,"698,504,259.1100","7,838.2122",0.7846,0.9153,"3,100.0000",51.9907,54.8929,46.7900,0.5619,-0.0362
0,0,8,21379556,"1,079,332,299.6300","7,301.7609",0.7011,0.9088,"2,673.7500",47.8244,42.9854,35.9787,0.4638,0.0072


,cluster_id,market_count,total_transactions,total_gmv_usd,avg_daily_transactions,transaction_growth_rate,completion_rate,active_services,supply_score,macro_potential_score,opportunity_score,rain_day_share,avg_rain_transaction_lift_pct,markets
0,1,4,16675254,"660,253,581.3500","11,390.2008",0.6958,0.9040,"3,943.2500",74.5172,19.5440,45.5975,0.6653,0.0534,"Bangalore, Jakarta, Manila, Mumbai"
1,2,1,3338833,"217,194,167.9400","9,122.4945",0.7005,0.9164,"3,537.0000",48.0147,71.5565,71.3800,0.9754,0.0469,Singapore
2,3,3,8606357,"698,504,259.1100","7,838.2122",0.7846,0.9153,"3,100.0000",51.9907,54.8929,46.7900,0.5619,-0.0362,"London, New York, Tokyo"
3,0,8,21379556,"1,079,332,299.6300","7,301.7609",0.7011,0.9088,"2,673.7500",47.8244,42.9854,35.9787,0.4638,0.0072,"Bangkok, Dubai, Ho Chi Minh City, Istanbul, Me..."


In [9]:
fig = px.scatter(
    clustered_markets,
    x="opportunity_score",
    y="avg_daily_transactions",
    color="cluster_id",
    size="total_gmv_usd",
    hover_name="market_name",
    hover_data=["nova_region", "opportunity_segment", "active_services", "transaction_growth_rate"],
    title="Market Clusters: Opportunity vs Demand Scale",
    labels={"opportunity_score": "Opportunity score", "avg_daily_transactions": "Avg daily transactions"},
)
fig.show()

fig = px.bar(
    cluster_profile.sort_values("avg_daily_transactions", ascending=True),
    x="avg_daily_transactions",
    y="cluster_id",
    orientation="h",
    color="cluster_id",
    title="Average Daily Transactions by Cluster",
    labels={"avg_daily_transactions": "Avg daily transactions", "cluster_id": "Cluster"},
)
fig.show()

## 10. Assign Analyst-Friendly Cluster Labels

The mapping below is intentionally editable. Review the cluster profiles first, then adjust the labels if a different wording fits the dashboard better.

In [10]:
# These labels are based on the current cluster profiles and should be reviewed after reruns.
cluster_label_map = {
    0: "Broad monitor markets",
    1: "High-demand supply-led markets",
    2: "Singapore: high-opportunity outlier",
    3: "Developed high-value growth markets",
}

cluster_label_map


{0: 'Broad monitor markets',
 1: 'High-demand supply-led markets',
 2: 'Singapore: high-opportunity outlier',
 3: 'Developed high-value growth markets'}

In [11]:
clustered_markets["cluster_label"] = clustered_markets["cluster_id"].map(cluster_label_map)
cluster_profile["cluster_label"] = cluster_profile["cluster_id"].map(cluster_label_map)
cluster_profile_with_members = cluster_profile_with_members.merge(
    cluster_profile[["cluster_id", "cluster_label"]],
    on="cluster_id",
    how="left",
)

cluster_assignments = clustered_markets[
    [
        "market_id",
        "market_name",
        "nova_region",
        "country_name",
        "country_iso3",
        "latitude",
        "longitude",
        "opportunity_score",
        "opportunity_segment",
        "cluster_id",
        "cluster_label",
    ]
].copy()

display(cluster_assignments.sort_values(["cluster_id", "market_name"]))
display(cluster_profile_with_members)

,market_id,market_name,nova_region,country_name,country_iso3,latitude,longitude,opportunity_score,opportunity_segment,cluster_id,cluster_label
3,4,Bangkok,Southeast Asia,Thailand,THA,13.7563,100.5018,41.3300,Monitor,0,Broad monitor markets
6,7,Dubai,EMEA,United Arab Emirates,ARE,25.2048,55.2708,48.0700,Maintain,0,Broad monitor markets
4,5,Ho Chi Minh City,Southeast Asia,Vietnam,VNM,10.8231,106.6297,28.3000,Monitor,0,Broad monitor markets
5,6,Istanbul,EMEA,Turkey,TUR,41.0082,28.9784,41.0300,Monitor,0,Broad monitor markets
10,11,Mexico City,Latin America,Mexico,MEX,19.4326,-99.1332,25.4800,Monitor,0,Broad monitor markets
7,8,Riyadh,EMEA,Saudi Arabia,SAU,24.7136,46.6753,40.4400,Monitor,0,Broad monitor markets
11,12,Sao Paulo,Latin America,Brazil,BRA,-23.5558,-46.6396,38.7100,Monitor,0,Broad monitor markets
15,16,Sydney,Oceania,Australia,AUS,-33.8688,151.2093,24.4700,Monitor,0,Broad monitor markets
13,14,Bangalore,South Asia,India,IND,12.9716,77.5946,39.5000,Monitor,1,High-demand supply-led markets
1,2,Jakarta,Southeast Asia,Indonesia,IDN,-6.2088,106.8456,53.5500,Maintain,1,High-demand supply-led markets


,cluster_id,market_count,total_transactions,total_gmv_usd,avg_daily_transactions,transaction_growth_rate,completion_rate,active_services,supply_score,macro_potential_score,opportunity_score,rain_day_share,avg_rain_transaction_lift_pct,markets,cluster_label
0,1,4,16675254,"660,253,581.3500","11,390.2008",0.6958,0.9040,"3,943.2500",74.5172,19.5440,45.5975,0.6653,0.0534,"Bangalore, Jakarta, Manila, Mumbai",High-demand supply-led markets
1,2,1,3338833,"217,194,167.9400","9,122.4945",0.7005,0.9164,"3,537.0000",48.0147,71.5565,71.3800,0.9754,0.0469,Singapore,Singapore: high-opportunity outlier
2,3,3,8606357,"698,504,259.1100","7,838.2122",0.7846,0.9153,"3,100.0000",51.9907,54.8929,46.7900,0.5619,-0.0362,"London, New York, Tokyo",Developed high-value growth markets
3,0,8,21379556,"1,079,332,299.6300","7,301.7609",0.7011,0.9088,"2,673.7500",47.8244,42.9854,35.9787,0.4638,0.0072,"Bangkok, Dubai, Ho Chi Minh City, Istanbul, Me...",Broad monitor markets


## 11. Interpret the Market Clusters

Cluster IDs are only model-assigned numbers. The business labels below are based on the cluster profile table, the market membership, and the features used for clustering. If the model is rerun with new data, review this section again before exporting to BigQuery.

### Cluster 0: Broad monitor markets

**Markets:** Bangkok, Dubai, Ho Chi Minh City, Istanbul, Mexico City, Riyadh, Sao Paulo, Sydney.

This is the largest cluster by market count, but not the strongest on a per-market basis. It has the lowest average daily transactions, the lowest active services, the lowest supply score, and the lowest opportunity score. The practical interpretation is a broad baseline group: these markets have meaningful demand, but they look less mature or less clearly prioritized than the other clusters. For the dashboard narrative, this group is useful as the comparison point for stronger demand, supply, and opportunity clusters.

### Cluster 1: High-demand supply-led markets

**Markets:** Bangalore, Jakarta, Manila, Mumbai.

This is the strongest current-demand cluster. It has the highest average daily transactions and the highest service supply measures, including active services and supply score. From a business perspective, these markets look supply-led: demand is already scaled, and the depth of available services appears closely connected to transaction volume. These are important operating markets where maintaining service availability likely matters for protecting demand.

### Cluster 2: Singapore: high-opportunity outlier

**Market:** Singapore.

Singapore forms a one-market cluster because its profile is distinct. It has the highest macro potential score and opportunity score, strong completion/reliability, and solid demand, but it does not have the same supply-heavy profile as Cluster 1. This should be interpreted as a strategic outlier rather than a normal multi-market segment. In the dashboard, it is clearer to call this out directly instead of hiding it inside a generic growth-market label.

### Cluster 3: Developed high-value growth markets

**Markets:** London, New York, Tokyo.

This cluster combines high GMV, strong completion rates, and the highest average transaction growth rate. These markets look like developed, high-value markets that are still growing. Their average rain transaction lift is negative, so weather does not appear to be the main demand driver here; in fact, rainy conditions may suppress or shift demand depending on the local category mix. The stronger story is high-value growth rather than weather sensitivity.

### Modeling caveat

The silhouette score was stronger for `k=3` than for `k=4`, so a three-cluster solution may be statistically cleaner. We keep `k=4` here because separating Singapore creates a useful business story and avoids forcing a distinct market into a broader segment. Since there are only 16 markets, treat these clusters as exploratory segments for analysis and dashboard storytelling, not as final operational rules.

### Questions to ask when presenting

1. Which markets already have scaled demand and deep service supply?
2. Which markets look like broad monitor markets rather than near-term growth priorities?
3. Does Singapore deserve a separate strategic treatment because of its opportunity profile?
4. Do developed high-value markets behave differently from high-demand supply-led markets?
5. Do the clusters tell a different story than the opportunity segments?


## 12. Ingest Cluster Results into BigQuery

This section writes the clustering outputs into `dbt_doruk` for Tableau.

Run this only after reviewing the cluster profiles and labels.

In [12]:
CLUSTER_MODEL_VERSION = "geo_market_kmeans_v1"
DESTINATION_DATASET = DATASET_ID
GENERATED_AT_UTC = pd.Timestamp.utcnow()


def add_export_metadata(data: pd.DataFrame) -> pd.DataFrame:
    exported = data.copy()
    exported["cluster_model_version"] = CLUSTER_MODEL_VERSION
    exported["generated_at_utc"] = GENERATED_AT_UTC
    return exported


cluster_assignments_export = add_export_metadata(cluster_assignments)
cluster_profiles_export = add_export_metadata(cluster_profile_with_members)
cluster_feature_inputs_export = add_export_metadata(feature_input)
cluster_diagnostics_export = add_export_metadata(k_results_df)

upload_tables = {
    "geo_market_cluster_assignments": cluster_assignments_export,
    "geo_market_cluster_profiles": cluster_profiles_export,
    "geo_market_cluster_feature_inputs": cluster_feature_inputs_export,
    "geo_market_cluster_diagnostics": cluster_diagnostics_export,
}

for table_name, upload_df in upload_tables.items():
    if upload_df.empty:
        print(f"Skipping {table_name}: no rows to upload")
        continue

    destination_table = f"{DESTINATION_DATASET}.{table_name}"
    print(f"Uploading {len(upload_df):,} rows to {PROJECT_ID}.{destination_table}")
    pandas_gbq.to_gbq(
        dataframe=upload_df,
        destination_table=destination_table,
        project_id=PROJECT_ID,
        if_exists="replace",
        location=LOCATION,
    )

print("BigQuery ingest complete.")

/var/folders/28/8k26_pcd2bz699b6mx6vz2yc0000gn/T/ipykernel_43308/2982138569.py:3: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  GENERATED_AT_UTC = pd.Timestamp.utcnow()


Uploading 16 rows to nova-project-498911.dbt_doruk.geo_market_cluster_assignments
Uploading 4 rows to nova-project-498911.dbt_doruk.geo_market_cluster_profiles
Uploading 16 rows to nova-project-498911.dbt_doruk.geo_market_cluster_feature_inputs
Uploading 3 rows to nova-project-498911.dbt_doruk.geo_market_cluster_diagnostics
BigQuery ingest complete.
